In [ ]:
import os, numpy as np, pandas as pd, seaborn as sns, matplotlib.pyplot as plt, json 

def plot_variant(results_list, prop_weights_list, S_t, xlim,
                 methods=("Weibull", "DCTB"),
                 label_style="plain",
                 patient=""):          # "plain"  oder  "math"
    """
    label_style:  "plain"  → |{w_i=0}|   (Text)
                  "math"   → |{w_i=0}|   (LaTeX/Math-Text)
    """

    # ---------- Label-Helfer --------------------------------------------------
    if label_style == "plain":
        row_label = lambda m, pw: f"{m}  |{{w_i=0}}|={pw/10:.1f}"
    elif label_style == "math":
        row_label = lambda m, pw: rf"{m}  $\lvert\{{w_i=0\}}\rvert={pw/10:.1f}$"
    else:
        raise ValueError("label_style muss 'plain' oder 'math' sein.")

    # ---------- DataFrame im Long-Format -------------------------------------
    frames = []
    for res, pw in zip(results_list, prop_weights_list):
        n = res["wb_pred"].shape[0]
        if "Weibull" in methods:
            frames.append(pd.DataFrame({
                "value": res["wb_pred"],
                "row"  : [row_label("Weibull", pw)] * n
            }))
        if "DCTB" in methods:
            frames.append(pd.DataFrame({
                "value": res["rf_pred"],
                "row"  : [row_label("DCTB", pw)] * n
            }))
    df = pd.concat(frames, ignore_index=True)

    # Reihenfolge der Zeilen
    row_order = []
    for m in ("Weibull", "DCTB"):
        if m in methods:
            row_order += [row_label(m, pw) for pw in prop_weights_list]

    # ---------- Plot ----------------------------------------------------------
    sns.set_style("whitegrid")
    fig, ax = plt.subplots(figsize=(7, 3 + 0.4*len(row_order)))

    sns.stripplot(data=df, x="value", y="row",
                  order=row_order, jitter=0.25,
                  alpha=0.5, size=1.5, color="teal", ax=ax)

    # Mittelwerte
    for i, r in enumerate(row_order):
        mean_val = df.loc[df["row"] == r, "value"].mean()
        ax.vlines(mean_val, i-0.4, i+0.4,
                  color="black", linestyle="--", linewidth=1)

    # Rote Linien nur in der jeweils passenden Zeile
    row_idx = {r: i for i, r in enumerate(row_order)}
    for pw, s in zip(prop_weights_list, S_t):
        for m in methods:
            lbl = row_label(m, pw)
            if lbl in row_idx:
                i = row_idx[lbl]
                ax.vlines(s, i-0.4, i+0.4, color="red", linewidth=1)

    ax.set_xlim(xlim)
    ax.set_ylabel("")
    if patient == "averageS":
        ax.set_xlabel(r"$\hat{\theta}(x_{avg})=\hat{P}\left(T^*>3 \mid X=x_{avg}\right)$")
    elif patient == "lowerS":
        ax.set_xlabel(r"$\hat{\theta}(x_{low})=\hat{P}\left(T^*>3 \mid X=x_{low}\right)$")
    elif patient == "higherS":
        ax.set_xlabel(r"$\hat{\theta}(x_{high})=\hat{P}\left(T^*>3 \mid X=x_{high}\right)$")
    fig.tight_layout()

    # Optional speichern
    if not os.path.exists("help_figures"):
        os.makedirs("help_figures")
    fig.savefig(f"help_figures/preds_{'_'.join(methods)}_{patient}.png",
                dpi=300)
    plt.close()

def plot_dctb(results_list,            # [results1, results3, results5]
              prop_weights_list,       # [1, 3, 5]
              S_t,                    # drei rote Referenzwerte
              xlim,                   # (xmin, xmax)
              label_style="plain",
              patient=""):   # "plain" oder "math"
    """
    Strip-Plot mit 3 Zeilen – nur DCTB.

    label_style:
        "plain" →  DCTB  (|{w_i=0}|=0.1)      ← robust
        "math"  →  DCTB  $|\{w_i=0\}|=0.1$    ← LaTeX-Look
    """

    # ---------- Label-Helfer --------------------------------------------------
    if label_style == "plain":
        row_label = lambda pw: f"(|{{w_i=0}}|={pw/10:.1f})"
    elif label_style == "math":
        row_label = lambda pw: "cens-level"+rf"$={pw/10:.1f}$"
    else:
        raise ValueError("label_style muss 'plain' oder 'math' sein.")

    # ---------- Daten zusammenführen -----------------------------------------
    frames = []
    for res, pw in zip(results_list, prop_weights_list):
        n = res["rf_pred"].shape[0]
        frames.append(pd.DataFrame({
            "value": res["rf_pred"],
            "row"  : [row_label(pw)] * n
        }))
    df = pd.concat(frames, ignore_index=True)

    row_order = [row_label(pw) for pw in prop_weights_list]   # 3 Zeilen

    # ---------- Plot ----------------------------------------------------------
    sns.set_style("whitegrid")
    fig, ax = plt.subplots(figsize=(7, 3 + 0.4*len(row_order)))

    sns.stripplot(data=df, x="value", y="row",
                  order=row_order, jitter=0.25,
                  alpha=0.5, size=1.5, color="teal", ax=ax)

    # Mittelwerte
    for i, r in enumerate(row_order):
        m = df.loc[df["row"] == r, "value"].mean()
        ax.vlines(m, i-0.4, i+0.4, color="black", linestyle="--", linewidth=1)

    # Rote Referenz-Linien
    for pw, s in zip(prop_weights_list, S_t):
        i = row_order.index(row_label(pw))
        ax.vlines(s, i-0.4, i+0.4, color="red", linewidth=1)

    ax.set_xlim(xlim)
    ax.set_ylabel("")
    if patient == "averageS":
        ax.set_xlabel(r"$\hat{\theta}(x_{avg})=\hat{P}\left(T^*>3 \mid X=x_{avg}\right)$")
    elif patient == "lowerS":
        ax.set_xlabel(r"$\hat{\theta}(x_{low})=\hat{P}\left(T^*>3 \mid X=x_{low}\right)$")
    elif patient == "higherS":
        ax.set_xlabel(r"$\hat{\theta}(x_{high})=\hat{P}\left(T^*>3 \mid X=x_{high}\right)$")
    fig.tight_layout()

    # Optional speichern
    if not os.path.exists("help_figures"):
        os.makedirs("help_figures")
    fig.savefig(f"help_figures/preds_dctb_{patient}.png", dpi=300)
    plt.close()

def coverage_prop(ijk_std_estimate, S_dach, S):

    u = S_dach - 1.96 * ijk_std_estimate
    o = S_dach + 1.96 * ijk_std_estimate

    prop = np.sum((u <= S) & (o >= S)) / len(u)
    return prop.round(2)

def plot_dctb_variances(results_list,        # [results1, results3, results5]
                        prop_weights_list,   # [1, 3, 5]
                        xlim2,               # (xmin, xmax)
                        patient=None,
                        label_style="math"   # "plain" oder "math"
                        ):
    """
    Plot mit 9 Zeilen  (3 Varianzschätzer × 3 |w_i=0|-Stufen)

        pro Stufe (0.1, 0.3, 0.5):
            1  V̂_{IJ-U}^{wB}
            2  V̂_{IJ-U}^{B}
            3  V̂_{Boot}

    rote Linie = std(rf_pred) der jeweiligen Stufe
    """

    # ---------- Label-Suffix  -------------------------------------------------
    if label_style == "plain":
        suf = lambda pw: f" (|{{w_i=0}}|={pw/10:.1f})"
    elif label_style == "math":
        suf = lambda pw: rf"$|\{{w_i=0\}}|={pw/10:.1f}$"
    else:
        raise ValueError("label_style muss 'plain' oder 'math' sein.")

    base = {
        "wB"   : r"$\hat{V}_{IJ-U}^{wB}$",
        "B"    : r"$\hat{V}_{IJ-U}^{B}$",
        "boot" : r"$\hat{V}_{Boot}$"
    }

    frames, cov_records, std_by_pw = [], [], {}

    # ---------- Daten einsammeln  --------------------------------------------
    for res, pw in zip(results_list, prop_weights_list):
        v_wB   = res["ijk_u_jahn_var"].clip(lower=0).pipe(np.sqrt)
        v_B    = res["ijk_u_wager_var"].clip(lower=0).pipe(np.sqrt)
        v_boot = res["boot_var"].clip(lower=0).pipe(np.sqrt)

        std_rf = res["rf_pred"].std(ddof=1)
        std_by_pw[pw] = std_rf

        cov_records.append({
            "|{w_i=0}|": pw/10,
            base["wB"]  : coverage_prop(v_wB,   res["rf_pred"], std_rf),
            base["B"]   : coverage_prop(v_B,    res["rf_pred"], std_rf),
            base["boot"]: coverage_prop(v_boot, res["rf_pred"], std_rf)
        })

        n = len(v_wB)
        frames.append(pd.DataFrame({
            "value": np.concatenate([v_wB, v_B, v_boot]),
            "row" : [f"{base['wB']}  {suf(pw)}"]  * n +
                    [f"{base['B']}   {suf(pw)}"]  * n +
                    [f"{base['boot']} {suf(pw)}"] * n,
            "pw"  : [pw] * 3 * n
        }))

    df_all = pd.concat(frames, ignore_index=True)

    # ---------- Zeilenreihenfolge 0.1 → 0.3 → 0.5 -----------------------------
    row_order = []
    for pw in prop_weights_list:            # 1, 3, 5
        row_order += [
            f"{base['wB']}  {suf(pw)}",
            f"{base['B']}   {suf(pw)}",
            f"{base['boot']} {suf(pw)}"
        ]

    # ---------- Plot  ---------------------------------------------------------
    sns.set_style("whitegrid")
    fig, ax = plt.subplots(figsize=(7, 3.5 + 0.5*len(row_order)))

    sns.stripplot(data=df_all, x="value", y="row",
                  order=row_order, jitter=0.25,
                  alpha=0.5, size=1.5, color="teal", ax=ax)

    # Mittelwert-Striche
    for i, r in enumerate(row_order):
        m = df_all.loc[df_all["row"] == r, "value"].mean()
        ax.vlines(m, i-0.4, i+0.4, color="black",
                  linestyle="--", linewidth=1)

    # Rote Linien (über 3 Zeilen jeder Stufe)
    for pw in prop_weights_list:
        std_val = std_by_pw[pw]
        top     = row_order.index(f"{base['wB']}  {suf(pw)}")
        bottom  = top + 2
        ax.vlines(std_val, top-0.4, bottom+0.4,
                  color="red", linewidth=1.2, zorder=10)

    # Achsen-Layout
    ax.set_xlim(xlim2)
    ax.set_ylabel("")
    xlabel_map = {
        "averageS": r"$\hat{\sigma}\!\left(\hat{\theta}(x_{avg})\right)$",
        "lowerS"  : r"$\hat{\sigma}\!\left(\hat{\theta}(x_{low})\right)$",
        "higherS" : r"$\hat{\sigma}\!\left(\hat{\theta}(x_{high})\right)$"
    }
    if patient in xlabel_map:
        ax.set_xlabel(xlabel_map[patient])
    fig.tight_layout()

    # Speichern
    if not os.path.exists("help_figures"):
        os.makedirs("help_figures")
    fig.savefig(f"help_figures/dctb_variances_{patient}.png", dpi=300)
    plt.close()

def plot_dctb_all_variances(results_list,        # [results1, results3, results5]
                            prop_weights_list,   # [1, 3, 5]
                            xlim2,               # (xmin, xmax)
                            patient=None,
                            label_style="math"   # "plain" oder "math"
                            ):
    """
    Plot mit 15 Zeilen:
        pro |w_i=0|-Stufe (0.1, 0.3, 0.5):
            1  V̂_IJ^{wB}      (biased)
            2  V̂_IJ-U^{wB}    (unbiased)
            3  V̂_IJ^{B}       (biased)
            4  V̂_IJ-U^{B}     (unbiased)
            5  V̂_Boot         (bootstrap)
    rote Linie = std(rf_pred) der jeweiligen Stufe
    """

    # ─── Label-Suffix ────────────────────────────────────────────────────────
    if label_style == "plain":
        suf = lambda pw: f" (|{{w_i=0}}|={pw/10:.1f})"
    elif label_style == "math":
        suf = lambda pw: rf"$|\{{w_i=0\}}|={pw/10:.1f}$"
    else:
        raise ValueError("label_style muss 'plain' oder 'math' sein.")

    base = {
        "wB_bias" : r"$\hat{V}_{IJ}^{wB}$",
        "wB"      : r"$\hat{V}_{IJ-U}^{wB}$",
        "B_bias"  : r"$\hat{V}_{IJ}^{B}$",
        "B"       : r"$\hat{V}_{IJ-U}^{B}$",
        "boot"    : r"$\hat{V}_{Boot}$"
    }

    frames, cov_records, std_by_pw = [], [], {}

    # ─── Daten einsammeln ────────────────────────────────────────────────────
    for res, pw in zip(results_list, prop_weights_list):
        v_wB_bias = res["ijk_jahn_var"].clip(lower=0).pipe(np.sqrt)
        v_wB      = res["ijk_u_jahn_var"].clip(lower=0).pipe(np.sqrt)
        v_B_bias  = res["ijk_wager_var"].clip(lower=0).pipe(np.sqrt)
        v_B       = res["ijk_u_wager_var"].clip(lower=0).pipe(np.sqrt)
        v_boot    = res["boot_var"].clip(lower=0).pipe(np.sqrt)

        std_rf = res["rf_pred"].std(ddof=1)
        std_by_pw[pw] = std_rf

        cov_records.append({
            "|{w_i=0}|": pw/10,
            base["wB_bias"]: coverage_prop(v_wB_bias, res["rf_pred"], std_rf),
            base["wB"]     : coverage_prop(v_wB,      res["rf_pred"], std_rf),
            base["B_bias"] : coverage_prop(v_B_bias,  res["rf_pred"], std_rf),
            base["B"]      : coverage_prop(v_B,       res["rf_pred"], std_rf),
            base["boot"]   : coverage_prop(v_boot,    res["rf_pred"], std_rf)
        })

        n = len(v_wB_bias)
        frames.append(pd.DataFrame({
            "value": np.concatenate([v_wB_bias, v_wB, v_B_bias, v_B, v_boot]),
            "row":  [f"{base['wB_bias']} {suf(pw)}"] * n +
                    [f"{base['wB']}      {suf(pw)}"] * n +
                    [f"{base['B_bias']}  {suf(pw)}"] * n +
                    [f"{base['B']}       {suf(pw)}"] * n +
                    [f"{base['boot']}    {suf(pw)}"] * n,
            "pw": [pw] * 5 * n
        }))

    df_all = pd.concat(frames, ignore_index=True)

    # ─── Zeilenreihenfolge 0.1 → 0.3 → 0.5 ──────────────────────────────────
    row_order = []
    for pw in prop_weights_list:
        row_order += [
            f"{base['wB_bias']} {suf(pw)}",
            f"{base['wB']}      {suf(pw)}",
            f"{base['B_bias']}  {suf(pw)}",
            f"{base['B']}       {suf(pw)}",
            f"{base['boot']}    {suf(pw)}"
        ]

    # ─── Plot ────────────────────────────────────────────────────────────────
    sns.set_style("whitegrid")
    fig, ax = plt.subplots(figsize=(7, 5 + 0.55*len(row_order)))

    sns.stripplot(data=df_all, x="value", y="row",
                  order=row_order, jitter=0.25,
                  alpha=0.5, size=1.5, color="teal", ax=ax)

    # Mittelwert-Striche
    for i, r in enumerate(row_order):
        m = df_all.loc[df_all["row"] == r, "value"].mean()
        ax.vlines(m, i-0.4, i+0.4, color="black",
                  linestyle="--", linewidth=1)

    # Rote Linien pro Stufe (über 5 Zeilen spannen)
    for pw in prop_weights_list:
        std_val = std_by_pw[pw]
        top    = row_order.index(f"{base['wB_bias']} {suf(pw)}")
        bottom = top + 4
        ax.vlines(std_val, top-0.4, bottom+0.4,
                  color="red", linewidth=1.2, zorder=10)

    # Achsen-Layout
    ax.set_xlim(xlim2)
    ax.set_ylabel("")
    xlabel_map = {
        "averageS": r"$\hat{\sigma}\!\left(\hat{\theta}(x_{\text{avg}})\right)$",
        "lowerS"  : r"$\hat{\sigma}\!\left(\hat{\theta}(x_{\text{low}})\right)$",
        "higherS" : r"$\hat{\sigma}\!\left(\hat{\theta}(x_{\text{high}})\right)$"
    }
    if patient in xlabel_map:
        ax.set_xlabel(xlabel_map[patient])
    fig.tight_layout()

    # Speichern
    if not os.path.exists("help_figures"):
        os.makedirs("help_figures")
    fig.savefig(f"help_figures/dctb_all_variances_{patient}.png", dpi=300)
    plt.close()



In [2]:
exp_save_path = r'C:\Users\rehan\meine_Repos\Masterarbeit\Paper\results\(n_train)1000__(B_RF)1000__(B_1)200__(n_sim)1000__(seed)42__4kovariates\121.90058'
patient = 'averageS'

#exp_save_path = r'C:\Users\rehan\meine_Repos\Masterarbeit\Paper\results\(n_train)1000__(B_RF)1000__(B_1)200__(n_sim)1000__(seed)42__4kovariates_higherS(tau)\109.90058'
#patient = 'higherS'

#exp_save_path = r'C:\Users\rehan\meine_Repos\Masterarbeit\Paper\results\(n_train)1000__(B_RF)1000__(B_1)200__(n_sim)1000__(seed)42__4kovariates_lowS(tau)\131.90058'
#patient = 'lowerS'


In [3]:
# erstelle verzeichnis, falls nicht vorhanden /help_figures
if not os.path.exists('help_figures'):
    os.makedirs('help_figures')

with open(exp_save_path + '/setting.json') as f:
    exp_settings = json.load(f)
S_t = exp_settings["true_survival_probability[1,3,5]"]

# lade results
results1 = pd.read_csv(exp_save_path + f"/results1__(zero_weights){exp_settings['portion_zero_weights_train[1,3,5]'][0]}__(seen_events){exp_settings['portion_seen_events_train[1,3,5]'][0]}.csv")
results3 = pd.read_csv(exp_save_path + f"/results3__(zero_weights){exp_settings['portion_zero_weights_train[1,3,5]'][1]}__(seen_events){exp_settings['portion_seen_events_train[1,3,5]'][1]}.csv")
results5 = pd.read_csv(exp_save_path + f"/results5__(zero_weights){exp_settings['portion_zero_weights_train[1,3,5]'][2]}__(seen_events){exp_settings['portion_seen_events_train[1,3,5]'][2]}.csv")
x_pred = exp_settings['X_pred_point']

#### Plots


In [4]:
plot_variant(
    results_list      = [results1, results3, results5],
    prop_weights_list = [1, 3, 5],
    S_t               = S_t,
    xlim              = (0.1,1),
    methods           = ("Weibull", "DCTB"),
    label_style       = "plain"  ,   #  ← robust,
    patient          = patient   #  ← averageS, lowerS, higherS
)

plot_dctb(
    results_list      = [results1, results3, results5],
    prop_weights_list = [1, 3, 5],
    S_t               = S_t,     # z. B. np.array([0.25, 0.40, 0.60])
    xlim              = (0.1,1),   # z. B. (0, 1)
    label_style       = "math",   # oder "plain"
    patient         = patient   # "averageS", "lowerS", "higherS"
    )

plot_dctb_variances(
    results_list      = [results1, results3, results5],
    prop_weights_list = [1, 3, 5],
    xlim2             = (0,0.4),
    patient          = patient,   # "averageS", "lowerS", "higherS"
    label_style       = "math"     # oder "plain"
)

plot_dctb_all_variances(
    results_list      = [results1, results3, results5],
    prop_weights_list = [1, 3, 5],
    xlim2             = (0,0.6),
    patient          = patient,   # "averageS", "lowerS", "higherS"
    label_style       = "math"     # oder "plain"
)

In [5]:
import numpy as np
import pandas as pd

# ------------------------------------------------------------------ #
# 1)  Coverage-Funktion (unverändert, nur als Referenz)              #
# ------------------------------------------------------------------ #
def coverage_prop(ijk_std_estimate, S_dach, S):
    """
    Anteil, in dem das 95-%-Intervall   S_dach ± 1.96·ijk_std_estimate
    den wahren Wert S einschließt.
    """
    u = S_dach - 1.96 * ijk_std_estimate
    o = S_dach + 1.96 * ijk_std_estimate
    prop = np.mean((u <= S) & (o >= S))
    return prop.round(2)


# ------------------------------------------------------------------ #
# 2)  Tabelle nur für wB, B, Boot                                    #
# ------------------------------------------------------------------ #
def coverage_table(results_list,              # [results1, results3, results5]
                   prop_weights_list,         # [1, 3, 5]
                   S_ref_values,              # [S0.1, S0.3, S0.5]  (skalar)
                   label_style="math"):       # "plain" od. "math"
    """
    Gibt eine LaTeX-Tabelle (und ein DataFrame) mit den Coverage-
    Wahrscheinlichkeiten für drei Schätzer zurück:
        V̂_{IJ-U}^{wB},  V̂_{IJ-U}^{B},  V̂_{Boot}
    Zeilen:  |{w_i=0}|=0.1, 0.3, 0.5
    """

    if len(results_list) != len(prop_weights_list) != len(S_ref_values):
        raise ValueError("Alle drei Listen müssen gleich lang sein.")

    # Labels (LaTeX-freundlich)
    if label_style == "plain":
        col_wB   = "V_IJ-U^wB"
        col_B    = "V_IJ-U^B"
        col_boot = "V_Boot"
    else:  # "math"
        col_wB   = r"$\hat{V}_{IJ-U}^{wB}$"
        col_B    = r"$\hat{V}_{IJ-U}^{B}$"
        col_boot = r"$\hat{V}_{Boot}$"

    rows = []
    for res, pw, S_ref in zip(results_list, prop_weights_list, S_ref_values):

        # √Varianzen (negatives kappen)
        v_wB   = res["ijk_u_jahn_var"]  .clip(lower=0).pipe(np.sqrt)
        v_B    = res["ijk_u_wager_var"] .clip(lower=0).pipe(np.sqrt)
        v_boot = res["boot_var"]        .clip(lower=0).pipe(np.sqrt)

        # Schätzer-Vektor (S_dach)      – hier: deine rf-Vorhersagen
        S_hat = res["rf_pred"]

        rows.append({
            r"$|\{w_i=0\}|$": pw / 10,
            col_wB  : coverage_prop(v_wB,   S_hat, S_ref),
            col_B   : coverage_prop(v_B,    S_hat, S_ref),
            col_boot: coverage_prop(v_boot, S_hat, S_ref)
        })

    table = (pd.DataFrame(rows)
               .set_index(r"$|\{w_i=0\}|$")
               .sort_index())

    print(table.to_latex(escape=False, float_format="%.2f"))
    return table

# Ergebnisse-Listen
results_list      = [results1, results3, results5]
prop_weights_list = [1, 3, 5]                 #  → 0.1, 0.3, 0.5

cov_df = coverage_table(results_list,
                        prop_weights_list,
                        S_t,
                        label_style="math")


\begin{tabular}{lrrr}
\toprule
 & $\hat{V}_{IJ-U}^{wB}$ & $\hat{V}_{IJ-U}^{B}$ & $\hat{V}_{Boot}$ \\
$|\{w_i=0\}|$ &  &  &  \\
\midrule
0.100000 & 0.93 & 0.92 & 0.96 \\
0.300000 & 0.96 & 0.94 & 0.96 \\
0.500000 & 0.97 & 0.94 & 0.97 \\
\bottomrule
\end{tabular}

